<a href="https://colab.research.google.com/github/sangeerthP-10/Sangeerth_P_Task_and_Assment_IN126008302_Innomatics/blob/main/IN126008302_GEN_AI/Task__05__Token_Classification_POS_Tagging_%26_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets seqeval

In [2]:
from datasets import load_dataset

dataset = load_dataset("conll2003")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using the latest cached version of the dataset since conll2003 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'conll2003' at /root/.cache/huggingface/datasets/conll2003/conll2003/1.0.0/9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98 (last modified on Mon Apr  6 13:24:05 2026).


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [3]:
dataset["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

label_list = dataset["train"].features["chunk_tags"].feature.names

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

In [5]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])  # first subword
            else:
                label_ids.append(-100)  # ignore other subwords

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [6]:
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

In [8]:
!pip install -U transformers

In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3
)

In [11]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [14]:
!pip install evaluate
import evaluate

metric = evaluate.load("seqeval")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [13]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],

    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [17]:
trainer.train()

Step,Training Loss
500,0.388863
1000,0.175038
1500,0.136777
2000,0.117768
2500,0.103510


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.17995354254859455, metrics={'train_runtime': 555.293, 'train_samples_per_second': 75.857, 'train_steps_per_second': 4.743, 'total_flos': 1050667520934456.0, 'train_loss': 0.17995354254859455, 'epoch': 3.0})

In [20]:
def predict(sentence):
    tokens = sentence.split()

    inputs = tokenizer(
        tokens,
        return_tensors="pt",
        is_split_into_words=True
    )

    # ✅ Move inputs to same device as model
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model(**inputs).logits
    predictions = outputs.argmax(dim=2)

    predicted_labels = [
        id2label[p.item()] for p in predictions[0]
    ]

    return list(zip(tokens, predicted_labels[1:len(tokens)+1]))

In [22]:
sentence = "John works at Google in California"
result = predict(sentence)

print("Input Sentence:")
print(sentence)
print("\nPredicted Chunk Tags:\n")

for word, tag in result:
    print(f"{word:12} -> {tag}")

Input Sentence:
John works at Google in California

Predicted Chunk Tags:

John         -> B-NP
works        -> B-VP
at           -> B-PP
Google       -> B-NP
in           -> B-PP
California   -> B-NP




## 🔍 Key Insights
- BERT captures context effectively, improving performance  
- Proper label alignment (`-100`) is essential  
- Chunking is harder than POS tagging as it works at phrase level  

---

## ⚠️ Challenges Faced
- Library version compatibility issues  
- CPU vs GPU mismatch errors  
- Handling subword tokenization  

---

## 📊 Observations
- Good performance on common patterns  
- Slight errors on rare/complex phrases  

---

## 🚀 Future Improvements
- Use advanced models like RoBERTa  
- Tune hyperparameters  
- Train on larger datasets  


---


## ✅ Conclusion
In this project, a BERT-based token classification model was built to perform chunking using the CoNLL-2003 dataset. The model was trained using Hugging Face Transformers with proper tokenization and label alignment.

The training loss decreased steadily (from ~0.38 to ~0.10), showing effective learning. The model successfully identifies phrase-level structures like noun phrases (NP), verb phrases (VP), and prepositional phrases (PP).

---

---

## 🎯 Final Remark
This project demonstrates the effectiveness of transformer models in solving token classification tasks in NLP.